In [60]:
print("OK")

OK


In [61]:
%pwd


'/Users/vedvyas.rauniyar/Documents/Project'

In [62]:
import os

In [63]:
os.chdir("../")

In [64]:
%pwd

'/Users/vedvyas.rauniyar/Documents'

In [65]:
import langchain
print(langchain.__version__)

1.4.6


In [66]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    DirectoryLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [67]:
#Extract Data from the PDF file
def load_pdf_file(data):
    loader = DirectoryLoader(data,
                             glob="*.pdf",
                             loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [68]:
extracted_data = load_pdf_file("/Users/vedvyas.rauniyar/Documents/Project/Medical-Chatboat-Gen-AI/Data")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNADH+Berkeley-BookItalic', '/FirstChar': 32, '/LastChar': 111, '/Subtype': '/Type1', '/FontDescriptor': IndirectObject(8788, 0, 4567556624), '/Widths': [241, 241, 352, 481, 481, 667, 778, 204, 315, 315, 500, 481, 241, 333, 241, 278, 481, 481, 481, 481, 481, 481, 481, 481, 481, 481, 241, 241, 481, 481, 481, 352, 800, 593, 519, 611, 685, 537, 519, 667, 685, 296, 296, 611, 500, 796, 685, 685, 519, 667, 556, 463, 556, 685, 593, 870, 593, 519, 556, 315, 278, 315, 481, 500, 241, 463, 444, 370, 463, 389, 259, 407, 463, 241, 241, 444, 241, 722, 481, 426]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNAFK+Berkeley-Bold', '/FirstChar': 32, '/

In [69]:
# Split the data into chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


In [70]:
text_chunks = text_split(extracted_data)
print("Length of text chunks:", len(text_chunks))

Length of text chunks: 5861


In [71]:
from langchain_huggingface import HuggingFaceEmbeddings

In [72]:
#Downloaed the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embedding = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
    return embedding

In [73]:
embeddings = download_hugging_face_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8942.34it/s]


In [74]:
query_result = embeddings.embed_query("Hello World")
print("Length", len(query_result))

Length 384


In [75]:
import os
print(os.getcwd())

/Users/vedvyas.rauniyar/Documents


In [76]:
import os

env_file = os.path.join(os.getcwd(), ".env")
print(env_file)
print(os.path.exists(env_file))

/Users/vedvyas.rauniyar/Documents/.env
False


In [77]:


from dotenv import load_dotenv

load_dotenv("/Users/vedvyas.rauniyar/Documents/Project/Medical-Chatboat-Gen-AI/.env")

import os
print(os.getenv("PINECONE_API_KEY"))

pcsk_2xm27f_U8ocq8xWAp7s9ao5Q8WrKJKpJsu3V4sZFHpSU4m4UWcLsggEPjpNNExyddqBsad


In [85]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [87]:
import certifi

print(certifi.where())


/Users/vedvyas.rauniyar/Documents/Project/Medical-Chatboat-Gen-AI/medicalchatboatenv/lib/python3.14/site-packages/certifi/cacert.pem


In [88]:
import os
print(os.path.exists(certifi.where()))



False


In [86]:
import traceback
from pinecone.grpc import PineconeGRPC as Pinecone

try:
    pc = Pinecone(api_key=PINECONE_API_KEY)
except Exception:
    traceback.print_exc()

Traceback (most recent call last):
  File "/var/folders/xj/mkdbp3rj7n532bxm8vlg8j180000gp/T/ipykernel_42126/3397759736.py", line 5, in <module>
    pc = Pinecone(api_key=PINECONE_API_KEY)
  File "/Users/vedvyas.rauniyar/Documents/Project/Medical-Chatboat-Gen-AI/medicalchatboatenv/lib/python3.14/site-packages/pinecone/_client.py", line 248, in __init__
  File "/Users/vedvyas.rauniyar/Documents/Project/Medical-Chatboat-Gen-AI/medicalchatboatenv/lib/python3.14/site-packages/pinecone/_internal/http_client.py", line 904, in __init__
  File "/Users/vedvyas.rauniyar/Documents/Project/Medical-Chatboat-Gen-AI/medicalchatboatenv/lib/python3.14/site-packages/httpx/_transports/default.py", line 153, in __init__
  File "/Users/vedvyas.rauniyar/Documents/Project/Medical-Chatboat-Gen-AI/medicalchatboatenv/lib/python3.14/site-packages/httpx/_config.py", line 40, in create_ssl_context
  File "/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/ssl.py", line 

In [84]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key= PINECONE_API_KEY)

index_name = "medicalbot"

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)


FileNotFoundError: [Errno 2] No such file or directory

In [82]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [83]:
# Embed each chunk and upsert the embeddings inot your pinecne index
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunks,
    index_name = index_name,
    embedding = embedding
)

ModuleNotFoundError: No module named 'langchain_pinecone'